# 🦅 Set Up

''' To use GPU
conda install -y pytorch pytorch-cuda=12.1 -c pytorch -c nvidia
'''

In [1]:
from functions import *

In [2]:
cd ..

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses


# 🦅 Prep Data
- Use the same features and data set

In [3]:
df_sg = pd.read_parquet("data/appliances_stage0_v1.parquet")

In [4]:
df_sg.head().iloc[:, :20]

,review_id,rating,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,text_words,main_category,average_rating,rating_number,price,store,categories,MPN_1,MPN_2,MPN_3,MPN_4
0,6,5.0,B00004YWK2,B00004YWK2,AFNA7RNBEH66UUMHPEE7XJFB3MYA,2023-01-03 12:37:29.824,1,True,175,33,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.037446,0.011010,0.022477,0.038484
1,5,5.0,B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,708,148,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.003132,-0.005602,0.004761,-0.021004
2,4,5.0,B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,115,24,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,-0.069358,-0.000819,0.006222,0.020757
3,3,1.0,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,47,9,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.031263,0.027401,0.002945,0.014352
4,2,1.0,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,47,9,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.054404,0.025682,0.012061,0.018683


In [5]:
df_with_sae = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding).ftr')

In [6]:
df_with_sae.head().iloc[:, :10]

,review_id,item_id,text,inter_review_time,irt_days,y_log_irt,sae_0,sae_1,sae_2,sae_3
0,2,B00004YWK2,Product is cheap and the door doesn't work well,8 days 02:00:26.779000,8.083643,2.206475,0.000000,1.425354,1.305985,0.000000
1,3,B00004YWK2,No cover to close it and plastic was separated.,2 days 09:35:40.733000,2.399777,1.223710,0.224967,1.231542,1.122654,0.101840
2,4,B00004YWK2,"I love this it keeps my garage, nice and warm ...",8 days 19:18:18.179000,8.804377,2.282829,0.226200,1.067915,1.358308,0.102711
3,5,B00004YWK2,"So I mounted this, as you see, behind and just...",26 days 03:04:48.890000,26.128344,3.300579,0.663035,1.612317,1.635834,0.299319
4,6,B00004YWK2,I have 3 if these and the 1st bought several y...,0 days 03:55:21.873000,0.163448,0.151388,0.477116,1.353411,1.241897,0.195181


In [7]:
df_sg.shape, df_with_sae.shape

((94297, 1306), (94297, 3078))

In [8]:
df_sg_mrgSAE = df_sg[['review_id', 'timestamp', 'rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number', 'y_hours', 'y_log', 'fold']].merge(
    df_with_sae.drop(columns=['inter_review_time', 'irt_days', 'y_log_irt']), how='left', on='review_id'
)

In [9]:
df_sg_mrgSAE.head().iloc[:, :20]

,review_id,timestamp,rating,helpful_vote,verified_purchase,text_len,text_words,average_rating,rating_number,y_hours,y_log,fold,item_id,text,sae_0,sae_1,sae_2,sae_3,sae_4,sae_5
0,6,2023-01-03 12:37:29.824,5.0,1,True,175,33,4.1,579,3.922743,1.593866,train,B00004YWK2,I have 3 if these and the 1st bought several y...,0.477116,1.353411,1.241897,0.195181,0.0,1.211486
1,5,2023-01-03 16:32:51.697,5.0,0,True,708,148,4.1,579,627.080247,6.442668,train,B00004YWK2,"So I mounted this, as you see, behind and just...",0.663035,1.612317,1.635834,0.299319,0.0,1.360516
2,4,2023-01-29 19:37:40.587,5.0,0,True,115,24,4.1,579,211.305050,5.358024,train,B00004YWK2,"I love this it keeps my garage, nice and warm ...",0.226200,1.067915,1.358308,0.102711,0.0,1.161993
3,3,2023-02-07 14:55:58.766,1.0,0,True,47,9,4.1,579,57.594648,4.070643,train,B00004YWK2,No cover to close it and plastic was separated.,0.224967,1.231542,1.122654,0.101840,0.0,0.752137
4,2,2023-02-10 00:31:39.499,1.0,0,True,47,9,4.1,579,194.007439,5.273038,train,B00004YWK2,Product is cheap and the door doesn't work well,0.000000,1.425354,1.305985,0.000000,0.0,0.901288


In [12]:
df_sg_mrgSAE.to_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding, meta).ftr')

# 🦅 Load Data

In [3]:
df_model = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding, meta).ftr')
df_model = df_model[df_model['sae_0'].notnull()]

In [4]:
df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})

C:\Users\chopi\AppData\Local\Temp\ipykernel_1979160\3091700506.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})


In [5]:
df_model.head().iloc[:, :20]

,review_id,timestamp,rating,helpful_vote,verified_purchase,text_len,text_words,average_rating,rating_number,y_hours,y_log,fold,item_id,text,sae_0,sae_1,sae_2,sae_3,sae_4,sae_5
0,6,2023-01-03 12:37:29.824,5.0,1,1,175,33,4.1,579,3.922743,1.593866,train,B00004YWK2,I have 3 if these and the 1st bought several y...,0.477116,1.353411,1.241897,0.195181,0.0,1.211486
1,5,2023-01-03 16:32:51.697,5.0,0,1,708,148,4.1,579,627.080247,6.442668,train,B00004YWK2,"So I mounted this, as you see, behind and just...",0.663035,1.612317,1.635834,0.299319,0.0,1.360516
2,4,2023-01-29 19:37:40.587,5.0,0,1,115,24,4.1,579,211.305050,5.358024,train,B00004YWK2,"I love this it keeps my garage, nice and warm ...",0.226200,1.067915,1.358308,0.102711,0.0,1.161993
3,3,2023-02-07 14:55:58.766,1.0,0,1,47,9,4.1,579,57.594648,4.070643,train,B00004YWK2,No cover to close it and plastic was separated.,0.224967,1.231542,1.122654,0.101840,0.0,0.752137
4,2,2023-02-10 00:31:39.499,1.0,0,1,47,9,4.1,579,194.007439,5.273038,train,B00004YWK2,Product is cheap and the door doesn't work well,0.000000,1.425354,1.305985,0.000000,0.0,0.901288


# 🦅 Prediction

In [6]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

import lightgbm as lgb

In [7]:
''' Feature definition '''
metadata_cols = ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number']
sae_cols = [c for c in df_model.columns if c.startswith('sae_')]
feature_cols = metadata_cols + sae_cols

target_col = 'y_log'

In [8]:
''' Train / Validation / Test split '''
df_train_valid = df_model[df_model['fold'].isin(['train', 'valid'])].copy()
df_test = df_model[df_model['fold'] == 'test'].copy()

X_train_valid = df_train_valid[feature_cols].copy()
y_train_valid = df_train_valid[target_col].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test[target_col].copy()

In [9]:
fold_map = {'train': -1, 'valid': 0}
test_fold = df_train_valid['fold'].map(fold_map).values

ps = PredefinedSplit(test_fold=test_fold)

In [10]:
del df_model, df_train_valid, df_test

## 🐔 Ridge

In [10]:
ridge = Ridge(random_state=42)

param_grid_ridge = {
    'alpha': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
}

ridge_grid = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid_ridge,
    scoring='neg_root_mean_squared_error',  # tuning on RMSE
    cv=ps,
    n_jobs=-1,
    refit=True,
    verbose=1
)

ridge_grid.fit(X_train_valid, y_train_valid)

best_ridge = ridge_grid.best_estimator_
print("Best Ridge params:", ridge_grid.best_params_)

Fitting 1 folds for each of 6 candidates, totalling 6 fits
Best Ridge params: {'alpha': 30.0}


In [12]:
# Evaluate on test
y_pred_ridge_test = best_ridge.predict(X_test)
ridge_rmse_test = mean_squared_error(y_test, y_pred_ridge_test)
ridge_mae_test = mean_absolute_error(y_test, y_pred_ridge_test)

print(f"[Ridge] Test RMSE: {ridge_rmse_test:.4f}")
print(f"[Ridge] Test MAE : {ridge_mae_test:.4f}")

[Ridge] Test RMSE: 2.6436
[Ridge] Test MAE : 1.2162


## 🐔 LGBM

In [11]:
lgb_reg = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1,
    #device='gpu'   # requires LightGBM with GPU support
)

param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.001, 0.0001],
    'max_depth': [10, 20, 30]
}

lgb_grid = GridSearchCV(
    estimator=lgb_reg,
    param_grid=param_grid_lgb,
    scoring='neg_root_mean_squared_error',  # tuning on RMSE
    cv=ps,
    n_jobs=1,
    refit=True,
    verbose=1
)

lgb_grid.fit(X_train_valid, y_train_valid)

best_lgb = lgb_grid.best_estimator_
print("Best LightGBM params:", lgb_grid.best_params_)

Fitting 1 folds for each of 27 candidates, totalling 27 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.229659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the train set: 85229, number of used features: 3071
[LightGBM] [Info] Start training from score 4.218596
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.185914 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the train set: 85229, number of used features: 3071
[LightGBM] [Info] Start training from score 4.218596
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.244994 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the tr

In [13]:
# Evaluate on test
y_pred_lgb_test = best_lgb.predict(X_test)
lgb_rmse_test = mean_squared_error(y_test, y_pred_lgb_test)
lgb_mae_test = mean_absolute_error(y_test, y_pred_lgb_test)

print(f"[LightGBM] Test RMSE: {lgb_rmse_test:.4f}")
print(f"[LightGBM] Test MAE : {lgb_mae_test:.4f}")

[LightGBM] Test RMSE: 2.3718
[LightGBM] Test MAE : 1.1720


# 🦅 Inspection

### 🐤 predict with RandomForest

In [63]:
df_with_sae = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding).ftr')

In [64]:
# Split (time-aware split is recommended)
m = int(0.7 * len(df_with_sae))
df_tr, df_va = df_with_sae.iloc[:m], df_with_sae.iloc[m:]

In [65]:
# Build review-level features and fit downstream model
F_tr = df_tr.drop(columns=['review_id', 'item_id', 'text', 'inter_review_time', 'irt_days', 'y_log_irt']).values
F_va = df_va.drop(columns=['review_id', 'item_id', 'text', 'inter_review_time', 'irt_days', 'y_log_irt']).values
texts_va = df_va['text'].values

In [67]:
model, metrics = fit_downstream_RF(F_tr, df_tr["y_log_irt"].values, F_va, df_va["y_log_irt"].values)
print(metrics)

{'R2': -0.0445678550961075, 'RMSE': 1.1396785278822599}


In [68]:
# 1) SHAP values on validation set
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(F_va)                 # shape: [n_samples, n_features]
mean_abs = np.abs(sv).mean(axis=0)                  # magnitude (importance)
mean_signed = sv.mean(axis=0)                       # direction on average (±)

order = np.argsort(mean_abs)[::-1]
topK = min(20, len(order))
sae_cols = [f"sae_f{i}" for i in range(model.n_features_in_)]

print("Top features by |SHAP| (validation):")
for idx in order[:topK]:
    direction = "↑ IRT (lower demand)" if mean_signed[idx] > 0 else (
                "↓ IRT (higher demand)" if mean_signed[idx] < 0 else "neutral")
    print(f"{idx:>5}  {sae_cols[idx]:<40}  |shap|={mean_abs[idx]:.5f}  mean_shap={mean_signed[idx]:+.5f}  {direction}")

Top features by |SHAP| (validation):
 1571  sae_f1571                                 |shap|=0.06269  mean_shap=-0.02138  ↓ IRT (higher demand)
 2852  sae_f2852                                 |shap|=0.03350  mean_shap=-0.01353  ↓ IRT (higher demand)
  204  sae_f204                                  |shap|=0.02672  mean_shap=+0.00143  ↑ IRT (lower demand)
  929  sae_f929                                  |shap|=0.02521  mean_shap=-0.00249  ↓ IRT (higher demand)
 2217  sae_f2217                                 |shap|=0.02377  mean_shap=-0.01025  ↓ IRT (higher demand)
 1234  sae_f1234                                 |shap|=0.01891  mean_shap=+0.00232  ↑ IRT (lower demand)
  942  sae_f942                                  |shap|=0.01260  mean_shap=-0.00612  ↓ IRT (higher demand)
 1891  sae_f1891                                 |shap|=0.01042  mean_shap=-0.00209  ↓ IRT (higher demand)
 1029  sae_f1029                                 |shap|=0.00941  mean_shap=-0.00032  ↓ IRT (higher demand)
 1

In [28]:
inspect_sae_feature(feature_idx=204, lm=lm, sae=sae, F_va=F_va, texts_va=texts_va, order=order, n_reviews_to_show=3)


Inspecting SAE Feature: 204

--- Top Example 1 (Review Index: 10545) ---
Review-Level Feature Activation: 2.0952

Highlighted Review (Feature Activation Points in RED):
If you have an espresso machine, you know how important it is to keep the machine clean inside and out.<br />This little screen is so simple, but it saves a lot of work.<br />I just drop the screen on top of my prepared puck and make my espresso as usual.<br />The screen prevents coffee from contacting the shower screen in my E61 group head.  This means that coffee isn't getting up into the group drain valve or the over-pressure valve.  ... and that means that these two valves won't get gummed up with coffee.  It also means that the shower head isn't collecting residual coffee.  In short,  the insides of my espresso machine remain clean.<br /><br />Does this little puck screen make better coffee?  Maybe.  Probably more consistent.  But to be honest my puck prep is pretty good and I don't get a lot of channeling.  But I